# Практика 14. Параметричні критерії: t-критерій для однієї та двох вибірок, порівняння дисперсій

**Варіант 1: Київ, Центр**

Мета: застосувати `ttest_1samp`, `ttest_ind`, `ttest_rel` і критерій Лівена `levene()` до власних даних та обґрунтувати вибір між стандартним і Welch-варіантом двовибіркового критерію.

In [1]:
import numpy as np
from scipy import stats

variant = 1
city = 'Київ'
region = 'Центр'
norm = 74.0
alpha = 0.05
kyiv_morning = np.array([84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85], dtype=float)

print(f'Варіант {variant}: {city}, регіон {region}')
print(f'Довідкова норма: {norm:.1f}%')
print('Ранкові значення:', kyiv_morning)

Варіант 1: Київ, регіон Центр
Довідкова норма: 74.0%
Ранкові значення: [84. 82. 78. 70. 65. 63. 61. 63. 68. 75. 81. 85.]


## Дані варіанта 1

Використано 12 місячних значень ранкової відносної вологості Києва. Норма — `74%`. Вечірні значення будуть побудовані пізніше з тієї самої ранкової вибірки.

In [2]:
t_one, p_one = stats.ttest_1samp(kyiv_morning, popmean=norm)

print(f'Середнє Києва: {kyiv_morning.mean():.4f}%')
print(f's Києва: {kyiv_morning.std(ddof=1):.4f} п.п.')
print(f't_one = {t_one:.4f}, p-value = {p_one:.6f}')
print('Рішення: відхиляємо H₀.' if p_one < alpha else 'Рішення: не відхиляємо H₀.')
print('Висновок: середня вологість статистично значуще відрізняється від норми 74%.' if p_one < alpha else 'Висновок: дані не дають підстав вважати середню вологість відмінною від норми 74%.')

Середнє Києва: 72.9167%
s Києва: 8.9489 п.п.
t_one = -0.4194, p-value = 0.683030
Рішення: не відхиляємо H₀.
Висновок: дані не дають підстав вважати середню вологість відмінною від норми 74%.


## Завдання 1. Одновибірковий t-критерій проти норми

Для Києва перевіряємо:

- **H₀:** `μ = 74%`, середня річна вологість відповідає довідковій нормі.
- **H₁:** `μ ≠ 74%`, середня річна вологість відрізняється від норми.

Тест двобічний, бо завдання питає про будь-яке відхилення.

In [3]:
manual_t = (kyiv_morning.mean() - norm) / (kyiv_morning.std(ddof=1) / np.sqrt(len(kyiv_morning)))
print(f'Ручна t-статистика = {manual_t:.8f}')
print(f't з ttest_1samp = {t_one:.8f}')
print(f'Збігаються з округленням? {np.isclose(manual_t, t_one)}')

Ручна t-статистика = -0.41935484
t з ttest_1samp = -0.41935484
Збігаються з округленням? True


## Завдання 2. Ручна перевірка t-статистики

Перевіряємо формулу `t = (x̄ - μ₀) / (s / √n)` без використання готового критерію.

In [4]:
odesa = np.array([80, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83])

levene_odesa_stat, levene_odesa_p = stats.levene(kyiv_morning, odesa)
use_equal_var_odesa = levene_odesa_p >= alpha
t_odesa, p_odesa = stats.ttest_ind(kyiv_morning, odesa, equal_var=use_equal_var_odesa)

print(f'Одеса: std = {odesa.std(ddof=1):.4f}')
print(f'Levene: statistic = {levene_odesa_stat:.4f}, p-value = {levene_odesa_p:.4f}')
print(f'equal_var = {use_equal_var_odesa}')
print(f't_ind = {t_odesa:.4f}, p-value = {p_odesa:.4f}')
print('Висновок: статистично значущої різниці середніх не виявлено.' if p_odesa >= alpha else 'Висновок: середні вологості статистично значуще відрізняються.')

Одеса: std = 5.4244
Levene: statistic = 7.2023, p-value = 0.0136
equal_var = False
t_ind = -0.6345, p-value = 0.5337
Висновок: статистично значущої різниці середніх не виявлено.


## Завдання 3. Незалежний двовибірковий критерій: Київ проти Одеси

Одеса належить до Південного регіону, тобто регіони різні. Спершу перевіряємо рівність дисперсій критерієм Лівена, а вже потім обираємо варіант `ttest_ind`.

In [5]:
lviv = np.array([85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86])

levene_lviv_stat, levene_lviv_p = stats.levene(kyiv_morning, lviv)
use_equal_var_lviv = levene_lviv_p >= alpha
t_lviv, p_lviv = stats.ttest_ind(kyiv_morning, lviv, equal_var=use_equal_var_lviv)

print(f'Львів: std = {lviv.std(ddof=1):.4f}')
print(f'Levene: statistic = {levene_lviv_stat:.4f}, p-value = {levene_lviv_p:.4f}')
print(f'equal_var = {use_equal_var_lviv}')
print(f't_ind = {t_lviv:.4f}, p-value = {p_lviv:.4f}')
print('Висновок: статистично значущої різниці середніх не виявлено.' if p_lviv >= alpha else 'Висновок: середні вологості статистично значуще відрізняються.')

Львів: std = 8.4437
Levene: statistic = 0.1290, p-value = 0.7229
equal_var = True
t_ind = -0.5162, p-value = 0.6109
Висновок: статистично значущої різниці середніх не виявлено.


## Завдання 4. Порівняння з протилежним висновком Лівена

Для протилежного результату беремо Львів із Західного регіону. Тут `p_Levene ≥ 0.05`, тому застосовуємо стандартний двовибірковий критерій із `equal_var=True`.

In [6]:
amplitude = np.array([3, 3, 4, 5, 6, 7, 7, 7, 6, 5, 4, 3])
evening_humidity = kyiv_morning - amplitude

t_rel, p_rel = stats.ttest_rel(kyiv_morning, evening_humidity)
differences = kyiv_morning - evening_humidity

print('Вечірні значення:', evening_humidity)
print(f'Середня різниця ранок - вечір: {differences.mean():.4f} п.п.')
print(f't_rel = {t_rel:.4f}, p-value = {p_rel:.8f}')
print(f'Рішення: {"відхиляємо H₀" if p_rel < alpha else "не відхиляємо H₀"}.')
print('Висновок: добова різниця вологості статистично значуща.' if p_rel < alpha else 'Висновок: статистично значущої добової різниці не виявлено.')

Вечірні значення: [81. 79. 74. 65. 59. 56. 54. 56. 62. 70. 77. 82.]
Середня різниця ранок - вечір: 5.0000 п.п.
t_rel = 10.8562, p-value = 0.00000032
Рішення: відхиляємо H₀.
Висновок: добова різниця вологості статистично значуща.


## Завдання 5. Парний t-критерій: ранок проти вечора

Вечірнє значення кожного місяця отримуємо відніманням однакової для всіх варіантів добової амплітуди. Це 12 пов'язаних пар вимірювань того самого місяця.

## Контрольні питання

**1. Чому t-, а не z-розподіл?** Генеральне стандартне відхилення вологості невідоме, тому його оцінюємо через `s`. Додаткова невизначеність заміни `σ` на `s` описується t-розподілом із `n - 1` ступенями свободи.

**2. Чому неправильний `equal_var` змінює p-value систематично?** Параметр визначає формулу стандартної похибки та ступені свободи. Якщо дисперсії різні, припущення pooled variance може спотворити знаменник і критичний розподіл, тому p-value може бути систематично завищеним або заниженим.

**3. Чому парний критерій потужніший для пар?** Він аналізує різниці всередині кожної пари. Спільні для ранкового й вечірнього вимірювання сезонні умови вилучаються, а розкид різниць зазвичай менший, ніж розкид окремих значень.

**4. Чи узгоджується Levene з порівнянням std?** Для Одеси формальний тест показав різні дисперсії, і її `std` справді помітно більший за київський. Для Львова `p_Levene = 0.7229 ≥ 0.05`, а стандартні відхилення близькі, що узгоджується з порівнянням.

## Підсумок

- `ttest_1samp` застосовано для порівняння середньої вологості Києва з довідковою нормою.
- Ручна формула підтвердила t-статистику одновибіркового критерію.
- Перед кожним `ttest_ind` виконано `levene()` і на його основі обрано `equal_var`.
- Для Одеси використано Welch-варіант, для Львова — стандартний варіант.
- `ttest_rel` використано для пов'язаних ранкових і вечірніх вимірювань Києва.